# Hugging face model

In [35]:
# import torchaudio
# import torch
# import soundfile as sf
# import numpy as np
# from transformers import AutoModel

# model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
# model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval().cuda()

# Hugging face embedding extraction code
This includes .wav to mel-spectrogram conversion

In [36]:


# source_file = "/path/to/input.wav"
# target_file = "/path/to/output.npy"
# target_length = 1024    # Recommended: 1024 for 10s audio
# norm_mean = -4.268
# norm_std = 4.569

# # Load and resample audio
# wav, sr = sf.read(source_file)
# waveform = torch.tensor(wav).float().cuda()
# if sr != 16000:
#     waveform = torchaudio.functional.resample(waveform, sr, 16000)

# # Normalize and convert to mel-spectrogram
# waveform = waveform - waveform.mean()
# mel = torchaudio.compliance.kaldi.fbank(
#     waveform.unsqueeze(0),
#     htk_compat=True,
#     sample_frequency=16000,
#     use_energy=False,
#     window_type='hanning',
#     num_mel_bins=128,
#     dither=0.0,
#     frame_shift=10
# ).unsqueeze(0)

# # Pad or truncate
# n_frames = mel.shape[1]
# if n_frames < target_length:
#     mel = torch.nn.ZeroPad2d((0, 0, 0, target_length - n_frames))(mel)
# else:
#     mel = mel[:, :target_length, :]

# # Normalize
# mel = (mel - norm_mean) / (norm_std * 2)
# mel = mel.unsqueeze(0).cuda()  # shape: [1, 1, T, F]

# # Extract features
# with torch.no_grad():
#     feat = model.extract_features(mel)

# feat = feat.squeeze(0).cpu().numpy()
# np.save(target_file, feat)
# print(f"Feature shape: {feat.shape}")
# print(f"Saved to: {target_file}")


# Unique classes extraction & labeling

In [37]:
import pandas as pd
import os
import glob

def get_dcase_num_classes(base_path):
    all_configs = []
    
    # Locate all CSV files in the dataset
    csv_files = glob.glob(os.path.join(base_path, "**/*.csv"), recursive=True)
    
    for file in csv_files:
        # Determine machine type from the folder structure
        machine_type = file.split(os.sep)[-2] 
        df = pd.read_csv(file)
        
        # IMPORTANT: Only look at training rows
        # We check if 'train' is in the file_name column
        train_df = df[df['file_name'].str.contains('train')].copy()
        
        for _, row in train_df.iterrows():
            file_name = row['file_name']
            
            # 1. Determine Domain (Source/Target) from filename
            domain = "source" if "source" in file_name else "target"
            
            # 2. Extract Attribute Values (columns like d1v, d2v, etc.)
            attr_cols = [col for col in train_df.columns if col.endswith('v')]
            attr_values = [str(row[col]) for col in attr_cols if pd.notna(row[col])]
            
            # 3. Create the unique label string
            if not attr_values or all(v == 'noAttribute' for v in attr_values):
                # Only Machine + Domain (e.g., "bearing_source")
                config_label = f"{machine_type}_{domain}"
            else:
                # Machine + Domain + Attributes (e.g., "fan_source_n_B")
                attrs_str = "_".join(attr_values)
                config_label = f"{machine_type}_{domain}_{attrs_str}"
            
            all_configs.append(config_label)

    # Use a set to find every unique combination encountered in the training data
    unique_classes = sorted(list(set(all_configs)))
    
    print(f"Total Unique Training Classes: {len(unique_classes)}")
    return unique_classes, len(unique_classes)

class DCASELabelEncoder:
    def __init__(self, unique_configs):
        # Create a dictionary mapping the string to an integer
        self.str_to_int = {label: i for i, label in enumerate(unique_configs)}
        # Create the reverse mapping (useful for debugging)
        self.int_to_str = {i: label for label in unique_configs for i, label in enumerate([label])}
        self.num_classes = len(unique_configs)

    def encode(self, machine, domain, attributes=None):
        # Ensure consistent formatting
        machine = machine.strip().lower()
        domain = domain.strip().lower()
        
        if not attributes or attributes == ['noAttribute']:
            label_str = f"{machine}_{domain}"
        else:
            # Clean each attribute
            clean_attrs = [str(a).strip() for a in attributes]
            attrs_str = "_".join(clean_attrs)
            label_str = f"{machine}_{domain}_{attrs_str}"
        
        # Debugging: Print the string being searched
        print(f"Searching for: '{label_str}'")
        
        if label_str not in self.str_to_int:
            print(f"Warning: {label_str} not found in encoder labels!")
            print(f"Available labels: {list(self.str_to_int.keys())[:]}")
            
        return self.str_to_int.get(label_str, -1)

In [38]:
# labels_list, _ = get_dcase_num_classes("data\\dcase2025t2\\dev_data\\raw\\")
# encoder = DCASELabelEncoder(labels_list)
# target_id = encoder.encode("valve", "source", ["0", "5"])
# target_id

# Create dataset with labels

In [39]:
import torch
import torchaudio
import pandas as pd
import os
from torch.utils.data import Dataset

class EATDCASEDataset(Dataset):
    def __init__(self, df, encoder, base_path, target_length=1024):
        self.df = df
        self.encoder = encoder
        self.base_path = base_path
        self.target_length = target_length
        # Norm values from EAT pre-training
        self.norm_mean = -4.268
        self.norm_std = 4.569

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Use the file_name from CSV to find the actual .wav file
        file_path = os.path.join(self.base_path, row['file_name'])
        
        # 1. Load Audio
        waveform, sr = torchaudio.load(file_path)
        
        # 2. Resample to 16kHz if necessary
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        
        # 3. Standardize and convert to Kaldi Mel-filterbank (Match EAT requirements)
        waveform = waveform - waveform.mean()
        mel = torchaudio.compliance.kaldi.fbank(
            waveform,
            htk_compat=True,
            sample_frequency=16000,
            use_energy=False,
            window_type='hanning',
            num_mel_bins=128,
            dither=0.0,
            frame_shift=10
        ) # Shape: [frames, 128]

        # 4. Pad or Truncate to fixed length (e.g., 1024 frames for 10s audio)
        n_frames = mel.shape[0]
        if n_frames < self.target_length:
            pad_amount = self.target_length - n_frames
            mel = torch.nn.functional.pad(mel, (0, 0, 0, pad_amount), "constant", 0)
        else:
            mel = mel[:self.target_length, :]

        # 5. Normalize using EAT pre-trained statistics
        mel = (mel - self.norm_mean) / (self.norm_std * 2)

        # 6. Get Target ID for ArcFace Classification
        # Extract attributes from row using the logic we built earlier
        file_name_parts = row['file_name'].split('/')
        machine = file_name_parts[0] 

        # Determine domain from the filename itself
        domain = "source" if "source" in row['file_name'] else "target"

        # Get attribute values from the row columns ending in 'v'
        attr_cols = [col for col in self.df.columns if col.endswith('v')]
        attr_values = [str(row[col]).strip() for col in attr_cols if pd.notna(row[col])]
        # Filter out 'noAttribute' to match your encoder's expectations
        attr_values = [v for v in attr_values if v.lower() != 'noattribute']

        # Use the encoder!
        target_id = self.encoder.encode(machine, domain, attr_values)

        # Return mel-spectrogram [T, F] and the label ID
        return mel, torch.tensor(target_id).long()

# Creating master df
contains all unique labels per file

In [40]:
import pandas as pd
import glob
import os

def create_master_dataframe(base_path):
    all_dfs = []
    # Find all training attribute CSVs across all machine folders
    csv_files = glob.glob(os.path.join(base_path, "**/*.csv"), recursive=True)
    
    for file in csv_files:
        df = pd.read_csv(file)
        # Filter to keep only the 'train' rows (no test/anomaly rows for training)
        train_only = df[df['file_name'].str.contains('train')].copy()
        all_dfs.append(train_only)
    
    # Combine them into one big table
    master_df = pd.concat(all_dfs, ignore_index=True)
    return master_df

# Usage
BASE_PATH = "data\\dcase2025t2\\dev_data\\raw\\"
master_df = create_master_dataframe(BASE_PATH)
print(f"Total training samples: {len(master_df)}")
# Check how many samples per machine type are included
master_df.tail()

Total training samples: 7000


,file_name,d1p,d1v,d2p,d2v,d3p,d3v
6995,valve/train/section_00_target_train_normal_000...,v1pat,5,v2pat,5,NaN,NaN
6996,valve/train/section_00_target_train_normal_000...,v1pat,5,v2pat,5,NaN,NaN
6997,valve/train/section_00_target_train_normal_000...,v1pat,5,v2pat,5,NaN,NaN
6998,valve/train/section_00_target_train_normal_000...,v1pat,5,v2pat,5,NaN,NaN
6999,valve/train/section_00_target_train_normal_000...,v1pat,5,v2pat,5,NaN,NaN


# Testing 

In [41]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader

# --- CONFIGURATION ---
BASE_PATH = r"data\dcase2025t2\dev_data\raw"
# Pick one CSV to test (e.g., bearing)
CSV_PATH = os.path.join(BASE_PATH, "valve", "attributes_00.csv") 

# 1. TEST THE LABEL ENCODER
print("Step 1: Testing Label Encoder...")
labels_list, _ = get_dcase_num_classes(BASE_PATH)
encoder = DCASELabelEncoder(labels_list)
print(f"Total classes found: {len(labels_list)}")

# 2. TEST DATASET LOADING
print("\nStep 2: Testing Dataset Loading...")
df = pd.read_csv(CSV_PATH)
# Only use training rows for the test
df_train = df[df['file_name'].str.contains('train')].copy()

dataset = EATDCASEDataset(
    df=df_train, 
    encoder=encoder, 
    base_path=BASE_PATH, 
    target_length=1024
)

try:
    # Try to get the very first item
    mel, label_id = dataset[0]
    
    print("✅ SUCCESS: Dataset loaded a sample!")
    print(f"Mel shape: {mel.shape}")      # Should be [1024, 128]
    print(f"Target ID: {label_id.item()}") # Should be an integer
    print(f"File tested: {df_train.iloc[0]['file_name']}")
    
except Exception as e:
    print("❌ FAILED: Error during loading.")
    print(f"Error details: {e}")
    
    # Debugging the path specifically
    sample_file = df_train.iloc[0]['file_name']
    full_path = os.path.join(BASE_PATH, sample_file)
    print(f"Looking for file at: {os.path.abspath(full_path)}")
    print(f"Does file exist? {os.path.exists(full_path)}")



Step 1: Testing Label Encoder...
Total Unique Training Classes: 51
Total classes found: 51

Step 2: Testing Dataset Loading...
Searching for: 'valve_source_0_5'
✅ SUCCESS: Dataset loaded a sample!
Mel shape: torch.Size([1024, 128])
Target ID: 46
File tested: valve/train/section_00_source_train_normal_0000_v1pat_00_v2pat_05.wav


In [42]:
# import matplotlib.pyplot as plt
# import torch

# # 1. Initialize your DataLoader (using the components we set up)
# test_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)

# # 2. Grab exactly one batch
# try:
#     batch_mel, batch_labels = next(iter(test_loader))
#     print(f"✅ Success! Batch loaded.")
#     print(f"Mel Tensor Shape: {batch_mel.shape}")   # Should be [4, 1024, 128]
#     print(f"Labels in Batch: {batch_labels.tolist()}")
# except Exception as e:
#     print(f"❌ Error loading batch: {e}")
#     # This usually means a file path in your CSV is wrong
#     raise e

# # 3. Visualize the Mel-spectrograms
# fig, axes = plt.subplots(1, 4, figsize=(20, 5))
# for i in range(4):
#     # .T (Transpose) if needed to get Time on X-axis and Freq on Y-axis
#     # We use .log2() for visualization because raw power scales are hard to see
#     img = batch_mel[i].cpu().numpy().T 
    
#     axes[i].imshow(img, origin='lower', aspect='auto', cmap='magma')
#     axes[i].set_title(f"Label ID: {batch_labels[i].item()}")
#     axes[i].set_xlabel("Time Frames")
#     axes[i].set_ylabel("Mel Bins")

# plt.tight_layout()
# plt.show()

# ArcFace Loss

In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcFaceLoss, self).__init__()
        self.s = s
        self.m = m
        # weight represents the 'center' of each of your 51 classes
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        # 1. Normalize features and weights to get cosine similarity
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # 2. Calculate the angle (theta)
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        
        # 3. Add the margin to the target class
        target_logit = torch.cos(theta + self.m)
        
        # 4. Replace original cosine with margined logit for the ground truth
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1.0)
        output = (one_hot * target_logit) + ((1.0 - one_hot) * cosine)
        
        # 5. Scale by 's' (temperature)
        return F.cross_entropy(output * self.s, label)

# Load model

In [44]:
import torch
import torch.nn as nn
from transformers import AutoModel
# Assuming you have the EAT library or hub model loaded
# from eat_model import EAT 

model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval().cuda()

class EATLoRAModel(nn.Module):
    def __init__(self, model_name="EAT-base", lora_rank=8):
        super().__init__()
        # 1. Load the pre-trained backbone
        self.backbone = torch.hub.load('microsoft/EAT', model_name) 
        
        # 2. Freeze the backbone weights
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # 3. Inject LoRA layers (Conceptual - depends on your LoRA implementation)
        # Most libraries like 'peft' can do this automatically:
        # self.backbone = get_peft_model(self.backbone, LoraConfig(r=lora_rank))
        
        # 4. Final Projection (EAT-base outputs a 768-dim vector)
        self.embedding_size = 768

    def forward(self, x):
        # x: [Batch, Time, Freq] -> [Batch, 1024, 128]
        # The backbone processes patches and returns a global embedding
        embedding = self.backbone.extract_features(x) 
        return embedding # This 768-dim vector goes into ArcFace

# Lora setup

In [45]:
from peft import LoraConfig, get_peft_model
import torchaudio
import torch
import soundfile as sf
import numpy as np
from transformers import AutoModel

model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval().cuda()

# 1. Define the LoRA Configuration
# We target the 'query' and 'value' projections in the Transformer blocks
config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["qkv", "proj"], 
    lora_dropout=0.05, 
    bias="none"
)

# 2. Wrap your EAT model
model = get_peft_model(model, config)
model.print_trainable_parameters() # This will show that <1% of params are trainable

trainable params: 450,560 || all params: 90,830,095 || trainable%: 0.4960


In [ ]:
import torch.nn as nn

class EATAnomalousTrainer(nn.Module):
    def __init__(self, eat_backbone):
        super().__init__()
        self.backbone = eat_backbone

    def forward(self, x):

        outputs = self.backbone(x)
    
        if isinstance(outputs, tuple):
            return outputs[0]
        return outputs

In [47]:
# 1. Setup Data
labels_list, _ = get_dcase_num_classes("data\\dcase2025t2\\dev_data\\raw\\")
encoder = DCASELabelEncoder(labels_list)
train_dataset = EATDCASEDataset(df=master_df, encoder=encoder, base_path=BASE_PATH)
loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. Setup Model & Loss
eat_lora = EATAnomalousTrainer(model).cuda()
# ArcFaceLoss holds the weights for your classes
criterion = ArcFaceLoss(in_features=768, out_features=encoder.num_classes).cuda()

# 3. Setup Optimizer
# We optimize the LoRA weights AND the ArcFace centers
optimizer = torch.optim.AdamW([
    {'params': eat_lora.parameters()},
    {'params': criterion.parameters()}
], lr=5e-5, weight_decay=0.1)

Total Unique Training Classes: 51


In [48]:
master_df.head()

,file_name,d1p,d1v,d2p,d2v,d3p,d3v
0,bearing/train/section_00_source_train_normal_0...,NaN,NaN,NaN,NaN,NaN,NaN
1,bearing/train/section_00_source_train_normal_0...,NaN,NaN,NaN,NaN,NaN,NaN
2,bearing/train/section_00_source_train_normal_0...,NaN,NaN,NaN,NaN,NaN,NaN
3,bearing/train/section_00_source_train_normal_0...,NaN,NaN,NaN,NaN,NaN,NaN
4,bearing/train/section_00_source_train_normal_0...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

SAVE_DIR = "checkpoints/eat_lora_system1"
os.makedirs(SAVE_DIR, exist_ok=True)

In [54]:
loader

In [53]:
# System-1 uses a linear decay or cosine scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)

# --- 3. The Training Loop ---
num_steps = 5
current_step = 0
running_loss = 0.0

print(f"Starting Training for {num_steps} steps...")
pbar = tqdm(total=num_steps, desc="Training System-1")

while current_step < num_steps:
    eat_lora.train()
    
    for mel, labels in loader:
        if current_step >= num_steps:
            break
            
        mel, labels = mel.cuda(), labels.cuda()
        
        # Forward
        # Shape of mel: [Batch, 1024, 128]
        embeddings = eat_lora(mel)
        loss = criterion(embeddings, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        # Logging
        running_loss += loss.item()
        current_step += 1
        pbar.update(1)
        
        # Periodically Save and Print
        if current_step % 500 == 0:
            avg_loss = running_loss / 500
            pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
            running_loss = 0.0
            
            # Save Checkpoint
            torch.save({
                'model_state_dict': eat_lora.state_dict(),
                'arcface_state_dict': criterion.state_dict(),
                'encoder': encoder,
                'step': current_step
            }, os.path.join(SAVE_DIR, f"checkpoint_step_{current_step}.pt"))

pbar.close()
print("Training Complete!")

Starting Training for 5 steps...


Training System-1:   0%|          | 0/5 [00:00<?, ?it/s]

Searching for: 'slider_source'
Searching for: 'bearing_source'
Searching for: 'valve_source_0_5'
Searching for: 'gearbox_source_B'
Searching for: 'valve_source_0_5'
Searching for: 'slider_source'
Searching for: 'valve_source_4_0'
Searching for: 'fan_source_A'
Searching for: 'slider_target'
Searching for: 'gearbox_source_A'
Searching for: 'slider_source'
Searching for: 'gearbox_source_B'


LibsndfileError: Error opening 'data\\dcase2025t2\\dev_data\\raw\\section_00_source_train_normal_0959_noAttribute': System error.